# 📥 Import ข้อมูลเข้า MySQL Database

**โปรเจค:** Portfolio Backtesting System  
**Database:** portfolio_backtesting  
**ข้อมูลทั้งหมด:** 208,899 rows

---

## 📋 ขั้นตอน:

1. ⚙️ ตั้งค่า MySQL connection (cell 1)
2. 🔌 เชื่อมต่อ MySQL (cell 2)
3. 📥 Import ข้อมูลทั้งหมด (cell 3)
4. ✅ ตรวจสอบผลลัพธ์ (cell 4)

**ใช้เวลา:** 2-3 นาที

**⚠️ สำคัญ:** ต้องรัน Cell ตามลำดับเท่านั้น!

---

## Cell 1: ตั้งค่า MySQL Connection

**แก้ไข password ให้ตรงกับเครื่องคุณ**

In [ ]:
# ========================================
# ตั้งค่า MySQL Connection
# ========================================

MYSQL_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': 'krittanut123456',  # 🔐 แก้ไข password ตรงนี้
    'database': 'portfolio_backtesting'
}

# ========================================
# ติดตั้ง libraries (ถ้ายังไม่มี)
# ========================================

try:
    import mysql.connector
    import pandas as pd
    from datetime import datetime
    print("✅ Libraries พร้อมใช้งาน")
except ImportError:
    print("⏳ กำลังติดตั้ง libraries...")
    import sys
    !{sys.executable} -m pip install mysql-connector-python pandas -q
    import mysql.connector
    import pandas as pd
    from datetime import datetime
    print("✅ ติดตั้ง libraries เสร็จแล้ว")

print(f"\n✅ ตั้งค่าเรียบร้อย")
print(f"   Connection: {MYSQL_CONFIG['user']}@{MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}/{MYSQL_CONFIG['database']}")

import os

# ========================================
# แสดง Working Directory
# ========================================
print(f"📂 Current directory: {os.getcwd()}\n")

# ========================================
# เชื่อมต่อ MySQL
# ========================================
print("🔌 กำลังเชื่อมต่อ MySQL...")
try:
    conn = mysql.connector.connect(**MYSQL_CONFIG)
    cursor = conn.cursor(dictionary=True)
    print("✅ เชื่อมต่อ MySQL สำเร็จ")
    
    # ตรวจสอบ tables
    cursor.execute("SHOW TABLES")
    tables = [list(t.values())[0] for t in cursor.fetchall()]
    print(f"✅ พบ {len(tables)} tables ใน database\n")
    
except mysql.connector.Error as e:
    print(f"❌ เชื่อมต่อ MySQL ไม่สำเร็จ: {e}")
    print("\n💡 กรุณาตรวจสอบ:")
    print("   1. MySQL server ทำงานอยู่หรือไม่?")
    print("   2. Password ถูกต้องหรือไม่? (แก้ใน Cell 1)")
    print("   3. Database 'portfolio_backtesting' มีอยู่หรือไม่?")
    raise

# ========================================
# ตรวจสอบไฟล์ CSV
# ========================================
print("📁 ตรวจสอบไฟล์ CSV:")
data_files = {
    'ETF List': 'data/etf_list.csv',
    'Benchmarks': 'data/benchmark_portfolios.csv',
    'Holdings': 'data/benchmark_holdings.csv',
    'Price History': 'data/etf_price_history.csv'
}

all_exists = True
missing_files = []

for name, path in data_files.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024  # MB
        rows = sum(1 for _ in open(path)) - 1  # นับ rows (ไม่รวม header)
        print(f"   ✅ {name:15s}: {path:35s} ({size:5.1f} MB, {rows:,} rows)")
    else:
        print(f"   ❌ {name:15s}: ไม่พบไฟล์ {path}")
        missing_files.append(path)
        all_exists = False

if not all_exists:
    print("\n" + "="*70)
    print("❌ ERROR: ไม่พบไฟล์ CSV บางไฟล์!")
    print("="*70)
    print(f"\n📂 Current directory: {os.getcwd()}")
    print(f"\n❌ ไฟล์ที่ไม่พบ:")
    for f in missing_files:
        print(f"   - {f}")
    print("\n💡 วิธีแก้:")
    print("   1. ปิด Jupyter Notebook")
    print("   2. เปิด Terminal/CMD")
    print("   3. cd ไปที่โฟลเดอร์ desktop-tutorial:")
    print("      cd /path/to/desktop-tutorial")
    print("   4. รัน Jupyter Notebook จากโฟลเดอร์นั้น:")
    print("      jupyter notebook import_data.ipynb")
    print("\nหรือ")
    print("   เพิ่มโค้ดนี้ใน Cell 1 (ก่อน import libraries):")
    print("   import os")
    print("   os.chdir('/path/to/desktop-tutorial')  # แก้ path ให้ถูกต้อง")
    print("="*70)
    raise FileNotFoundError("ไม่พบไฟล์ CSV บางไฟล์")

print("\n🎉 พร้อม Import ข้อมูล!")

In [ ]:
import os

# เชื่อมต่อ MySQL
print("🔌 กำลังเชื่อมต่อ MySQL...\n")
try:
    conn = mysql.connector.connect(**MYSQL_CONFIG)
    cursor = conn.cursor(dictionary=True)
    print("✅ เชื่อมต่อ MySQL สำเร็จ")
    
    # ตรวจสอบ tables
    cursor.execute("SHOW TABLES")
    tables = [list(t.values())[0] for t in cursor.fetchall()]
    print(f"✅ พบ {len(tables)} tables ใน database")
    
except mysql.connector.Error as e:
    print(f"❌ เชื่อมต่อ MySQL ไม่สำเร็จ: {e}")
    print("\n💡 กรุณาตรวจสอบ:")
    print("   1. MySQL server ทำงานอยู่หรือไม่?")
    print("   2. Password ถูกต้องหรือไม่? (แก้ใน Cell 1)")
    print("   3. Database 'portfolio_backtesting' มีอยู่หรือไม่?")
    raise

# ตรวจสอบไฟล์ CSV
print("\n📁 ตรวจสอบไฟล์ CSV:")
data_files = {
    'ETF List': 'data/etf_list.csv',
    'Benchmarks': 'data/benchmark_portfolios.csv',
    'Holdings': 'data/benchmark_holdings.csv',
    'Price History': 'data/etf_price_history.csv'
}

all_exists = True
for name, path in data_files.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024  # MB
        rows = sum(1 for _ in open(path)) - 1  # นับ rows (ไม่รวม header)
        print(f"   ✅ {name:15s}: {path:35s} ({size:5.1f} MB, {rows:,} rows)")
    else:
        print(f"   ❌ {name:15s}: ไม่พบไฟล์ {path}")
        all_exists = False

if not all_exists:
    raise FileNotFoundError("ไม่พบไฟล์ CSV บางไฟล์")

print("\n🎉 พร้อม Import ข้อมูล!")

---

## Cell 3: Import ข้อมูลทั้งหมด

**⚠️ รันครั้งเดียวเท่านั้น!** ถ้ารันซ้ำจะเกิด duplicate error

ใช้เวลา **2-3 นาที**

In [ ]:
def import_all_data():
    """Import ข้อมูลทั้งหมดเข้า MySQL"""
    
    start_time = datetime.now()
    print("🚀 เริ่ม Import ข้อมูล...\n")
    print("="*70)
    
    try:
        # ========================================
        # 1. Import etf_master (50 rows)
        # ========================================
        print("\n📊 [1/4] Import ETF Master...")
        df_etf = pd.read_csv('data/etf_list.csv')
        df_etf = df_etf.sort_values('ticker_symbol').reset_index(drop=True)
        print(f"   📁 อ่านไฟล์: {len(df_etf)} rows")
        
        for _, row in df_etf.iterrows():
            cursor.execute("""
                INSERT INTO etf_master 
                (ticker_symbol, etf_name, asset_class, region, sector, expense_ratio, inception_date)
                VALUES (%s, %s, %s, %s, %s, %s, %s)
            """, (
                row['ticker_symbol'],
                row['etf_name'],
                row['asset_class'],
                row['region'],
                row['sector'],
                float(row['expense_ratio']),
                row['inception_date']
            ))
        
        conn.commit()
        print(f"   ✅ Import สำเร็จ: {len(df_etf)} ETFs")
        
        # สร้าง mapping: ticker → etf_id
        cursor.execute("SELECT etf_id, ticker_symbol FROM etf_master")
        etf_map = {r['ticker_symbol']: r['etf_id'] for r in cursor.fetchall()}
        print(f"   📋 สร้าง mapping: {len(etf_map)} tickers → IDs")
        
        # ========================================
        # 2. Import benchmark_portfolios (35 rows)
        # ========================================
        print("\n📊 [2/4] Import Benchmark Portfolios...")
        df_bench = pd.read_csv('data/benchmark_portfolios.csv')
        df_bench = df_bench.sort_values('benchmark_name').reset_index(drop=True)
        print(f"   📁 อ่านไฟล์: {len(df_bench)} rows")
        
        for _, row in df_bench.iterrows():
            cursor.execute("""
                INSERT INTO benchmark_portfolios 
                (benchmark_name, description, risk_level, target_return, asset_allocation)
                VALUES (%s, %s, %s, %s, %s)
            """, (
                row['benchmark_name'],
                row['description'],
                row['risk_level'],
                float(row['target_return']),
                row['asset_allocation']
            ))
        
        conn.commit()
        print(f"   ✅ Import สำเร็จ: {len(df_bench)} benchmarks")
        
        # สร้าง mapping: benchmark_name → benchmark_id
        cursor.execute("SELECT benchmark_id, benchmark_name FROM benchmark_portfolios")
        bench_map = {r['benchmark_name']: r['benchmark_id'] for r in cursor.fetchall()}
        print(f"   📋 สร้าง mapping: {len(bench_map)} benchmarks → IDs")
        
        # ========================================
        # 3. Import benchmark_holdings (116 rows)
        # ========================================
        print("\n📊 [3/4] Import Benchmark Holdings...")
        df_holdings = pd.read_csv('data/benchmark_holdings.csv')
        print(f"   📁 อ่านไฟล์: {len(df_holdings)} rows")
        
        imported = 0
        for _, row in df_holdings.iterrows():
            benchmark_id = bench_map.get(row['benchmark_name'])
            etf_id = etf_map.get(row['ticker_symbol'])
            
            if benchmark_id and etf_id:
                cursor.execute("""
                    INSERT INTO benchmark_holdings (benchmark_id, etf_id, target_weight)
                    VALUES (%s, %s, %s)
                """, (benchmark_id, etf_id, float(row['target_weight'])))
                imported += 1
        
        conn.commit()
        print(f"   ✅ Import สำเร็จ: {imported} holdings")
        
        # ========================================
        # 4. Import price_history (208,700 rows)
        # ========================================
        print("\n📊 [4/4] Import Price History (ใช้เวลา 1-2 นาที)...")
        df_price = pd.read_csv('data/etf_price_history.csv')
        print(f"   📁 อ่านไฟล์: {len(df_price):,} rows")
        
        # เตรียมข้อมูล
        df_price['etf_id'] = df_price['ticker'].map(etf_map)
        df_price = df_price.dropna(subset=['etf_id'])
        print(f"   🔄 แปลง ticker → etf_id: {len(df_price):,} rows")
        
        # Import เป็น batch (เร็วกว่า)
        batch_size = 5000
        total_batches = (len(df_price) + batch_size - 1) // batch_size
        
        imported_price = 0
        for i in range(0, len(df_price), batch_size):
            batch = df_price.iloc[i:i+batch_size]
            
            # Prepare data
            data = [
                (
                    int(row['etf_id']),
                    row['date'],
                    float(row['open']),
                    float(row['high']),
                    float(row['low']),
                    float(row['close']),
                    int(row['volume'])
                )
                for _, row in batch.iterrows()
            ]
            
            # Batch insert
            cursor.executemany("""
                INSERT INTO price_history 
                (etf_id, price_date, open_price, high_price, low_price, close_price, volume)
                VALUES (%s, %s, %s, %s, %s, %s, %s)
            """, data)
            
            imported_price += len(batch)
            current_batch = i // batch_size + 1
            percent = (imported_price / len(df_price)) * 100
            print(f"   ⏳ Progress: {imported_price:,}/{len(df_price):,} rows ({percent:.1f}%) - Batch {current_batch}/{total_batches}", end='\r')
        
        conn.commit()
        print(f"\n   ✅ Import สำเร็จ: {imported_price:,} price records                    ")
        
        # ========================================
        # สรุปผลลัพธ์
        # ========================================
        elapsed = (datetime.now() - start_time).total_seconds()
        print("\n" + "="*70)
        print("\n🎉 Import เสร็จสมบูรณ์!")
        print(f"⏱️  ใช้เวลา: {elapsed:.1f} วินาที ({elapsed/60:.1f} นาที)")
        print("\n📊 สรุปผลลัพธ์:")
        print(f"   ✅ ETF Master:           {len(df_etf):>10,} rows")
        print(f"   ✅ Benchmark Portfolios: {len(df_bench):>10,} rows")
        print(f"   ✅ Benchmark Holdings:   {imported:>10,} rows")
        print(f"   ✅ Price History:        {imported_price:>10,} rows")
        print(f"   {'─'*40}")
        print(f"   📦 Total:                {len(df_etf) + len(df_bench) + imported + imported_price:>10,} rows")
        print("\n" + "="*70)
        
    except mysql.connector.IntegrityError as e:
        print(f"\n\n⚠️  IntegrityError: {e}")
        print("\n💡 Tip: มีข้อมูลอยู่แล้ว! ถ้าต้องการ import ใหม่ รัน SQL นี้ก่อน:")
        print("""
        DELETE FROM price_history;
        DELETE FROM benchmark_holdings;
        DELETE FROM benchmark_portfolios;
        DELETE FROM etf_master;
        """)
        raise
    except Exception as e:
        print(f"\n\n❌ Error: {e}")
        import traceback
        traceback.print_exc()
        raise

# รัน import
import_all_data()

---

## Cell 4: ตรวจสอบผลลัพธ์

เช็คว่า import ครบถ้วนและแสดงตัวอย่างข้อมูล

In [ ]:
print("\n✅ ตรวจสอบข้อมูลใน Database:\n")
print("="*70)

tables_expected = {
    'etf_master': 50,
    'benchmark_portfolios': 35,
    'benchmark_holdings': 114,
    'price_history': 208700
}

all_ok = True
for table, expected in tables_expected.items():
    cursor.execute(f"SELECT COUNT(*) as cnt FROM {table}")
    result = cursor.fetchone()
    count = result['cnt']
    
    # ตรวจสอบว่าได้อย่างน้อย 95% ของที่คาดหวัง
    threshold = expected * 0.95
    status = "✅" if count >= threshold else "⚠️"
    if count < threshold:
        all_ok = False
    
    print(f"{status} {table:25s}: {count:>10,} rows (expected ~{expected:,})")

print("="*70)

if all_ok:
    print("\n🎉 ตรวจสอบเสร็จสิ้น - ทุกอย่างถูกต้อง!")
else:
    print("\n⚠️  บางตารางมีข้อมูลน้อยกว่าที่คาดหวัง กรุณาตรวจสอบ")

# ========================================
# แสดงตัวอย่างข้อมูล
# ========================================

print("\n\n📋 ตัวอย่างข้อมูล ETF Master:")
print("="*70)
cursor.execute("SELECT * FROM etf_master LIMIT 5")
df_sample = pd.DataFrame(cursor.fetchall())
if not df_sample.empty:
    print(df_sample[['etf_id', 'ticker_symbol', 'etf_name', 'asset_class']].to_string(index=False))

print("\n\n📋 ตัวอย่างข้อมูล Benchmark Portfolios:")
print("="*70)
cursor.execute("SELECT * FROM benchmark_portfolios LIMIT 5")
df_sample = pd.DataFrame(cursor.fetchall())
if not df_sample.empty:
    print(df_sample[['benchmark_id', 'benchmark_name', 'risk_level', 'target_return']].to_string(index=False))

print("\n\n📋 ตัวอย่างข้อมูล Price History:")
print("="*70)
cursor.execute("""
    SELECT ph.*, e.ticker_symbol 
    FROM price_history ph 
    JOIN etf_master e ON ph.etf_id = e.etf_id 
    LIMIT 5
""")
df_sample = pd.DataFrame(cursor.fetchall())
if not df_sample.empty:
    print(df_sample[['ticker_symbol', 'price_date', 'close_price', 'volume']].to_string(index=False))

print("\n\n" + "="*70)
print("🎉 พร้อมใช้งาน!")
print("="*70)
print("\nขั้นตอนถัดไป:")
print("  1. เปิด main.ipynb เพื่อทดสอบ Portfolio Backtesting")
print("  2. เปิด analytics.ipynb เพื่อวิเคราะห์ผลลัพธ์")
print("  3. ใช้ database.ipynb สำหรับ CRUD operations")

---

## Cell 5: ปิดการเชื่อมต่อ (Optional)

รันเมื่อเสร็จสิ้นทุกอย่างแล้ว

In [ ]:
# ปิดการเชื่อมต่อ
cursor.close()
conn.close()
print("✅ ปิดการเชื่อมต่อ MySQL แล้ว")

---

## 💡 Tips และ Troubleshooting

### ถ้าต้องการลบข้อมูลและ import ใหม่:

```sql
DELETE FROM price_history;
DELETE FROM benchmark_holdings;
DELETE FROM benchmark_portfolios;
DELETE FROM etf_master;
```

จากนั้นรัน Cell 3 ใหม่

### ถ้า MySQL connection error:

1. ตรวจสอบ MySQL server ทำงานหรือไม่
2. ตรวจสอบ username/password ใน Cell 1
3. ตรวจสอบว่าสร้าง database `portfolio_backtesting` แล้วหรือไม่

### ถ้าไม่พบไฟล์ CSV:

- ตรวจสอบว่า Jupyter Notebook เปิดอยู่ที่โฟลเดอร์ `desktop-tutorial`
- ตรวจสอบว่ามีโฟลเดอร์ `data/` และไฟล์ CSV ครบ

### ถ้า import ช้าเกินไป:

- Price history (208,700 rows) ใช้เวลา 1-2 นาทีเป็นปกติ
- ถ้าช้ากว่านี้ ลองเพิ่ม `batch_size` ใน Cell 3 จาก 5000 เป็น 10000